# 10.4 Policy search and policy gradients: practice notebook

Skip learning values and tune the policy itself. Use softmax policies, the policy gradient theorem and REINFORCE with a baseline, then speed up with actor critic methods, and learn from experts with imitation and inverse RL.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### The softmax policy and its log gradient

In [ ]:
# A softmax policy over two actions and the gradient of log pi, which REINFORCE needs.
import math
theta = [0.0, 1.0]                          # one preference per action

def policy(theta):
    e = [math.exp(t) for t in theta]
    total = sum(e)
    return [x / total for x in e]

pi = policy(theta)
print("pi =", [round(p, 3) for p in pi])
for a in [0, 1]:
    grad = [(1 if k == a else 0) - pi[k] for k in range(2)]   # d log pi(a) / d theta_k
    print("action", a, "gradient of log pi:", [round(g, 3) for g in grad])

### REINFORCE on a two armed bandit

In [ ]:
# REINFORCE on a two armed bandit. Arm 0 pays 1 with chance 0.3, arm 1 pays 1 with chance 0.7.
import math, random
rng = random.Random(2)
theta = [0.0, 0.0]
alpha, baseline = 0.4, 0.0

for t in range(1, 41):
    e0, e1 = math.exp(theta[0]), math.exp(theta[1])
    pi = [e0 / (e0 + e1), e1 / (e0 + e1)]
    a = 0 if rng.random() < pi[0] else 1
    r = 1.0 if rng.random() < [0.3, 0.7][a] else 0.0
    for k in range(2):                                  # theta += alpha (r - b) grad log pi(a)
        theta[k] = theta[k] + alpha * (r - baseline) * ((1 if k == a else 0) - pi[k])
    baseline = baseline + (r - baseline) / t            # running average reward
print("theta:", [round(x, 3) for x in theta])
print("chance of the better arm:", round(math.exp(theta[1]) / (math.exp(theta[0]) + math.exp(theta[1])), 3))

## Your turn

### Task 1: A softmax policy

Write `softmax(theta)` that returns the list e^θᵢ ÷ Σ e^θ.

Hint: Exponentiate each preference, then divide by the total.

In [ ]:
import math

def softmax(theta):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('[round(p, 3) for p in softmax([0.0, 1.0])]', [0.269, 0.731]), ('softmax([0.0, 0.0])', [0.5, 0.5]), ('[round(p, 3) for p in softmax([1.0, 2.0, 3.0])]', [0.09, 0.245, 0.665])])

### Task 2: The gradient of log π

Write `grad_log_pi(theta, a)` that returns the list of ∂ log π(a) ÷ ∂θ_k = (1 if k = a else 0) − π(k), for a softmax policy.

Hint: Compute the softmax first, then 1 − π(a) for the chosen action and −π(k) for the others.

In [ ]:
import math

def grad_log_pi(theta, a):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('[round(g, 3) for g in grad_log_pi([0.0, 1.0], 0)]', [0.731, -0.731]), ('grad_log_pi([0.0, 0.0], 1)', [-0.5, 0.5]), ('round(sum(grad_log_pi([0.3, -1.2, 2.0], 2)), 6)', 0.0)])

### Task 3: REINFORCE on a two armed bandit

Write `reinforce(steps, alpha, seed)`: arm 0 pays 1 with chance 0.3 and arm 1 with chance 0.7 (else 0). Start at θ = (0, 0), pick arms from the softmax with `rng.random()`, update θ with the running average reward as the baseline, and return the final chance of arm 1.

Hint: Inside the loop: softmax, pick an arm, draw a reward, update both θ values, then update the baseline.

In [ ]:
import math, random

def reinforce(steps, alpha, seed):
    rng = random.Random(seed)
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('reinforce(2000, 0.1, 1) > 0.9', True), ('reinforce(2000, 0.1, 7) > 0.9', True), ('0.4 < reinforce(1, 0.1, 3) < 0.6', True)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: A softmax policy

Adding the same number to every θ gives the same policy.

In [ ]:
import math

def softmax(theta):
    e = []
    for t in theta:
        e.append(math.exp(t))
    total = sum(e)
    out = []
    for x in e:
        out.append(x / total)
    return out

In [ ]:
check([('[round(p, 3) for p in softmax([0.0, 1.0])]', [0.269, 0.731]), ('softmax([0.0, 0.0])', [0.5, 0.5]), ('[round(p, 3) for p in softmax([1.0, 2.0, 3.0])]', [0.09, 0.245, 0.665])])

### Solution to task 2: The gradient of log π

Test 3: the parts always add up to 0.

In [ ]:
import math

def grad_log_pi(theta, a):
    e = [math.exp(t) for t in theta]
    total = sum(e)
    pi = [x / total for x in e]
    out = []
    for k in range(len(theta)):
        if k == a:
            out.append(1 - pi[k])
        else:
            out.append(-pi[k])
    return out

In [ ]:
check([('[round(g, 3) for g in grad_log_pi([0.0, 1.0], 0)]', [0.731, -0.731]), ('grad_log_pi([0.0, 0.0], 1)', [-0.5, 0.5]), ('round(sum(grad_log_pi([0.3, -1.2, 2.0], 2)), 6)', 0.0)])

### Solution to task 3: REINFORCE on a two armed bandit

After 2000 pulls the policy picks the better arm more than 90 percent of the time.

In [ ]:
import math, random

def reinforce(steps, alpha, seed):
    rng = random.Random(seed)
    theta = [0.0, 0.0]
    baseline = 0.0
    for t in range(1, steps + 1):
        e0 = math.exp(theta[0])
        e1 = math.exp(theta[1])
        pi = [e0 / (e0 + e1), e1 / (e0 + e1)]
        if rng.random() < pi[0]:
            a = 0
        else:
            a = 1
        if rng.random() < [0.3, 0.7][a]:
            r = 1.0
        else:
            r = 0.0
        for k in range(2):
            if k == a:
                g = 1 - pi[k]
            else:
                g = -pi[k]
            theta[k] = theta[k] + alpha * (r - baseline) * g
        baseline = baseline + (r - baseline) / t
    e0 = math.exp(theta[0])
    e1 = math.exp(theta[1])
    return e1 / (e0 + e1)

In [ ]:
check([('reinforce(2000, 0.1, 1) > 0.9', True), ('reinforce(2000, 0.1, 7) > 0.9', True), ('0.4 < reinforce(1, 0.1, 3) < 0.6', True)])